# Rstats: one line, a full report

Synthetic teaching data. Install this checkout with `pip install -e .`. In Colab, install the repository after it is available on GitHub, then upload the two example CSVs.

In [ ]:
%matplotlib inline
from pathlib import Path
import pandas as pd
import rstats as rs

examples = Path('examples') if Path('examples/students.csv').exists() else Path('.')
df = pd.read_csv(examples / 'students.csv')
long_df = pd.read_csv(examples / 'longitudinal.csv')
long_df['time'] = pd.Categorical(long_df['time'], categories=['baseline', 'midterm', 'final'], ordered=True)
df.head()

## 1. Describe and tabulate
Missingness and denominators are part of the report.

In [ ]:
rs.descriptives(data=df, columns=['score', 'study_hours'], by='group')

In [ ]:
rs.table('group ~ passed', data=df, percent='row')

## 2. Categorical association
Look at expected cell counts as well as the p value.

In [ ]:
rs.chisq('group ~ passed', data=df)

## 3. ANOVA and ANCOVA
ANOVA predictors are groups. Covariates are continuous adjustments. `True` requests Tukey; other methods are named explicitly.

In [ ]:
rs.aov('score ~ group', data=df, posthocifsig=True)

In [ ]:
rs.aov('score ~ group', data=df, covariates=['study_hours'], posthocifsig='holm')

In [ ]:
rs.aov('score ~ group', data=df, method='welch', posthocifsig='games-howell')

In [ ]:
rs.aov('score ~ time * group', data=long_df, subject='student_id', within='time', posthocifsig='holm')

## 4. Linear and logistic models
Text groups are categorical automatically. Numeric-coded groups can use `categorical=['group']`. Logistic output identifies the modeled event.

In [ ]:
rs.lm('score ~ study_hours + group', data=df)

In [ ]:
rs.logmodel('passed ~ study_hours + group', data=df)

## 5. Plot independently
These commands return Matplotlib axes; customize labels or save with `ax.figure.savefig(...)`.

In [ ]:
rs.vis('score ~ study_hours', data=df, hue='group')

In [ ]:
rs.vis('score ~ group', data=df, type='box')

In [ ]:
rs.vis('score ~ time', data=long_df, type='line', estimator='mean', hue='group')

## Reuse the actual tables
Formatting never discards numerical precision. A significant association alone does not establish causation.

In [ ]:
result = rs.lm('score ~ study_hours + group', data=df)
result.tables['coefficients']